# Exploratory Data Analysis (EDA) with Spark

Unexecuted notebook. Run on remote Jupyter/Colab with Java 17, Spark 3.5.7 and requirements-remote.txt installed. See docs/EXECUTION.md. Upload source and real bank.csv. Save the executed notebook with outputs.

1. **Load the bank.csv**: src/eda.py via spark_job.py eda; 01_preview.csv and printed schema. Use real bank.csv, not bank-additional.csv.
2. **Perform basic data inspection**: 01_preview.csv, 01_numeric_summary.csv, and 01_quality.json show rows, numeric summaries, and missing values. Capture printed schema.
3. **Filter the dataset**: 02_balance_over_threshold.csv contains a bounded preview of balance>1000. Change --balance-threshold if needed; source DataFrame filtering processes all rows.
4. **Extract the month**: 02_04_06_engineered_columns.csv maps month abbreviations to month_number and ceil(month_number/3) as quarter. There is no full contact date/year.
5. **Group the data by job**: 03_job_balance_age.csv contains average balance and approximate median age (percentile_approx accuracy 10000). Explain the approximation.
6. **Find the total number**: 03_subscribers_by_marital.csv counts y=yes by marital status.
7. **Write a UDF**: banklib.age_group is registered as a Spark Python UDF: age<30, 30<=age<=60, age>60; null age becomes unknown. Engineered output demonstrates its use.
8. **Calculate the subscription rate**: 05_education_subscription.csv contains record counts and subscription percentages by education.
9. **Identify the top 3**: 05_top3_credit_default_jobs.csv ranks observed existing-credit-default rates. Unknown default statuses are excluded from the rate denominator. Loan-specific defaults cannot be established from this field.
10. **Concatenate the**: 02_04_06_engineered_columns.csv includes job_marital using concat_ws.
11. **Use string functions**: 02_04_06_engineered_columns.csv includes contact_upper using upper().
12. **Convert the Spark DataFrame**: export() calls toPandas only on bounded samples and small aggregates, rather than collecting an unbounded dataset. This fulfills conversion while limiting driver memory.
13. **Create a bar plot**: 07_job_counts.png and corresponding CSV show clients by job.
14. **Analyze which month**: 08_month_contacts.csv and 08_busiest_month.csv include contacts and success percentage.
15. **Find the average duration**: 08_duration_by_subscription.csv reports mean contact seconds, grouped by y.
16. **Calculate the correlation coefficient**: 09_age_balance_correlation.json reports Pearson r on complete age/balance pairs. Discuss direction and strength from the actual output.
17. **Analyze the proportion**: 10_credit_default_counts.csv contains default counts and proportions; 10_credit_default_counts.png plots counts. Keep unknown as a separate category.
18. **Investigate which contact method**: 11_contact_success.csv ranks subscription success percent by contact, with sample sizes. Observational differences are not causal effects.
19. **Calculate the average account balance**: 12_sql_age_balance.csv comes from a Spark temporary view and SQL GROUP BY age_group. Include the SQL and observed table.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if not (REPO / 'src').exists():
    REPO = REPO.parent
assert (REPO / 'src').exists(), 'Set REPO to your uploaded repository root'
sys.path.insert(0, str(REPO / 'src'))
INPUT = REPO / 'data' / 'bank.csv'
OUTPUT = REPO / 'artifacts' / 'notebook-eda-01'
EVENT_DIR = None  # optional existing event-log URI on your compute host
assert INPUT.is_file()
assert not OUTPUT.exists(), 'Choose a new output path'
OUTPUT.mkdir(parents=True)
from spark_common import session, load_bank, Monitor, save_json
spark = session('notebook-eda', partitions=2, event_dir=EVENT_DIR, master='local[2]')
spark.sparkContext.addPyFile(str(REPO / 'src' / 'banklib.py'))
save_json(OUTPUT / 'run_config.json', {'input': str(INPUT), 'spark_version': spark.version,
          'master': spark.sparkContext.master, 'seed_split': 2026, 'fold_hash_seed': 91})
print(spark.version)


In [ ]:
data = load_bank(spark, str(INPUT))
data.printSchema()
data.show(10)
data.groupBy('y').count().show()


In [ ]:
from eda import run
with Monitor(OUTPUT / 'resource_samples.json'):
    run(data, OUTPUT)


In [ ]:
for path in sorted(OUTPUT.glob('*.json')):
    print(path.name)
    print(path.read_text()[:12000])
from IPython.display import display, Image
for path in sorted(OUTPUT.glob('*.png')):
    display(Image(filename=str(path)))


## Observations to complete after execution
Record actual values, explain patterns and limitations, and reference saved artifacts.
Parallelism: include medians, CPU/RSS peaks, executor metrics and overhead.
Modeling: compare PR-AUC/recall and majority baseline, discuss imbalance and leakage.


In [ ]:
spark.stop()
